<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/Solucion_modelado_dimensional_ecommerce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelado dimensional con Pandas · E-commerce
## Solución guiada de los cuatro ejercicios

**Objetivo:** identificar hechos y dimensiones, relacionar tablas y calcular ventas, beneficio bruto y gasto por cliente.

**Cómo usarlo en Google Colab:** sube este archivo `.ipynb` y selecciona **Entorno de ejecución → Ejecutar todas**. No necesitas archivos externos ni instalar paquetes: solo utilizamos Pandas y las herramientas de visualización de tablas de Colab.

Las celdas incluyen resultados ya calculados. Puedes modificar los datos y volver a ejecutar todo para experimentar.

**Convenciones:** los importes se expresan en euros. Interpretamos `coste` como coste unitario; no incluimos impuestos, devoluciones, descuentos ni otros gastos.

## 0. Datos originales
Conservamos exactamente los datos del enunciado: 100 pedidos, 250 líneas, 30 productos y 20 clientes. Los datos son sintéticos y siguen patrones regulares; sirven para practicar operaciones, no para extraer conclusiones comerciales reales.

In [ ]:
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 12)
pd.set_option('display.float_format', '{:,.2f}'.format)

pedidos = pd.DataFrame({
    'pedido_id': range(1, 101),
    'cliente_id': [i % 20 + 1 for i in range(100)],
    'fecha': pd.date_range('2024-01-01', periods=100, freq='D'),
    'total': [round(50 + i * 1.5, 2) for i in range(100)]
})

lineas_pedido = pd.DataFrame({
    'linea_id': range(1, 251),
    'pedido_id': [i % 100 + 1 for i in range(250)],
    'producto_id': [i % 30 + 1 for i in range(250)],
    'cantidad': [1 + i % 5 for i in range(250)],
    'precio_unitario': [10 + (i % 30) * 2 for i in range(250)]
})

productos = pd.DataFrame({
    'producto_id': range(1, 31),
    'nombre': [f'Producto_{i}' for i in range(1, 31)],
    'categoria': ['Electrónica', 'Ropa', 'Hogar'] * 10,
    'coste': [5 + i for i in range(30)]
})

clientes = pd.DataFrame({
    'cliente_id': range(1, 21),
    'nombre': [f'Cliente_{i}' for i in range(1, 21)],
    'ciudad': ['Málaga', 'Sevilla', 'Granada', 'Córdoba'] * 5,
    'segmento': ['Premium', 'Standard', 'Básico', 'Premium'] * 5
})

for nombre, tabla in [('pedidos', pedidos), ('lineas_pedido', lineas_pedido),
                      ('productos', productos), ('clientes', clientes)]:
    print(f'{nombre}: {len(tabla)} filas')
    display(tabla.head())

pedidos: 100 filas
lineas_pedido: 250 filas
productos: 30 filas
clientes: 20 filas


,pedido_id,cliente_id,fecha,total
0,1,1,2024-01-01,50.00
1,2,2,2024-01-02,51.50
2,3,3,2024-01-03,53.00
3,4,4,2024-01-04,54.50
4,5,5,2024-01-05,56.00


,linea_id,pedido_id,producto_id,cantidad,precio_unitario
0,1,1,1,1,10
1,2,2,2,2,12
2,3,3,3,3,14
3,4,4,4,4,16
4,5,5,5,5,18


,producto_id,nombre,categoria,coste
0,1,Producto_1,Electrónica,5
1,2,Producto_2,Ropa,6
2,3,Producto_3,Hogar,7
3,4,Producto_4,Electrónica,8
4,5,Producto_5,Ropa,9


,cliente_id,nombre,ciudad,segmento
0,1,Cliente_1,Málaga,Premium
1,2,Cliente_2,Sevilla,Standard
2,3,Cliente_3,Granada,Básico
3,4,Cliente_4,Córdoba,Premium
4,5,Cliente_5,Málaga,Premium


## 1. ¿Cuál es la tabla de hechos? ¿Cuáles son las dimensiones?

| Tabla | Papel en este análisis | Granularidad: qué representa cada fila |
|---|---|---|
| `lineas_pedido` | Hechos de ventas con detalle | Una línea de pedido |
| `pedidos` | Cabecera que aporta cliente y fecha; también permite analizar hechos a nivel de pedido | Un pedido |
| `productos` | Dimensión de producto | Un producto |
| `clientes` | Dimensión de cliente | Un cliente |

**La granularidad elegida es una línea de pedido.** `linea_id` identifica cada hecho y `pedido_id` puede repetirse porque un pedido contiene varias líneas.

Relaciones:
- Un cliente tiene muchos pedidos.
- Un pedido tiene muchas líneas.
- Un producto puede aparecer en muchas líneas.

Las medidas de la línea son `cantidad`, `precio_unitario` y los importes calculados. El precio unitario no se suma como los ingresos.

Estas tablas son un modelo de origen con cabeceras y líneas, no un esquema estrella completo. En un Data Warehouse podríamos construir `FACT_VENTAS` a nivel de línea y conectarla directamente con `DIM_PRODUCTO`, `DIM_CLIENTE` y `DIM_FECHA`. Aquí utilizamos la fecha del pedido sin crear una dimensión temporal separada.

### Comprobación previa de claves y relaciones
Una unión puede multiplicar filas si la tabla de la derecha tiene claves duplicadas. Además, una referencia sin correspondencia podría perderse en una unión interna. Comprobamos ambas situaciones antes de continuar.

In [ ]:
assert pedidos['pedido_id'].is_unique
assert lineas_pedido['linea_id'].is_unique
assert productos['producto_id'].is_unique
assert clientes['cliente_id'].is_unique

assert lineas_pedido['pedido_id'].isin(pedidos['pedido_id']).all()
assert lineas_pedido['producto_id'].isin(productos['producto_id']).all()
assert pedidos['cliente_id'].isin(clientes['cliente_id']).all()
print('Claves únicas y todas las referencias tienen correspondencia.')

Claves únicas y todas las referencias tienen correspondencia.


## Preparación común: una tabla con contexto y medidas
`merge` relaciona tablas mediante una clave, igual que un `JOIN` de SQL.

1. Añadimos cliente y fecha a cada línea usando `pedido_id`.
2. Añadimos los atributos del producto usando `producto_id`.
3. Calculamos ingresos, coste y beneficio bruto de cada línea.

`validate='many_to_one'` comprueba que varias filas de la izquierda puedan apuntar a una sola fila de la derecha. No comprueba por sí solo que todas las referencias existan; eso se comprobó en la celda anterior.

Renombramos el nombre del producto para distinguirlo del nombre del cliente. Cada fila de `ventas` seguirá siendo una línea de pedido.

In [ ]:
ventas = (
    lineas_pedido
    .merge(
        pedidos[['pedido_id', 'cliente_id', 'fecha']],
        on='pedido_id', validate='many_to_one'
    )
    .merge(
        productos.rename(columns={'nombre': 'nombre_producto'}),
        on='producto_id', validate='many_to_one'
    )
)

ventas['importe_venta'] = ventas['cantidad'] * ventas['precio_unitario']
ventas['importe_coste'] = ventas['cantidad'] * ventas['coste']
ventas['beneficio_bruto'] = ventas['importe_venta'] - ventas['importe_coste']
ventas['mes'] = ventas['fecha'].dt.to_period('M')

assert len(ventas) == len(lineas_pedido)
print(f'Tabla de análisis: {len(ventas)} líneas de pedido.')
display(ventas.head())

Tabla de análisis: 250 líneas de pedido.


,linea_id,pedido_id,producto_id,cantidad,precio_unitario,cliente_id,...,categoria,coste,importe_venta,importe_coste,beneficio_bruto,mes
0,1,1,1,1,10,1,...,Electrónica,5,10,5,5,2024-01
1,2,2,2,2,12,2,...,Ropa,6,24,12,12,2024-01
2,3,3,3,3,14,3,...,Hogar,7,42,21,21,2024-01
3,4,4,4,4,16,4,...,Electrónica,8,64,32,32,2024-01
4,5,5,5,5,18,5,...,Ropa,9,90,45,45,2024-01


### Atención: el total de la cabecera no coincide con sus líneas
El campo `pedidos.total` se generó con una fórmula independiente. Por eso no representa necesariamente la suma de `cantidad × precio_unitario` de sus líneas.

Comprobamos la diferencia sin modificar los datos originales. Para los ejercicios 2 y 3 utilizaremos los importes de las líneas. En el ejercicio 4 mostraremos las dos interpretaciones.

Tampoco debemos sumar el total de la cabecera después de unirlo a las líneas: se repetiría una vez por línea y podría contar varias veces el importe de un pedido.

In [ ]:
totales_lineas = (
    ventas.groupby('pedido_id', as_index=False)
    .agg(total_lineas=('importe_venta', 'sum'))
)
comparacion_totales = pedidos[['pedido_id', 'total']].merge(
    totales_lineas, on='pedido_id', validate='one_to_one'
)
comparacion_totales['diferencia'] = (
    comparacion_totales['total'] - comparacion_totales['total_lineas']
).round(2)

print('Pedidos con diferencia:', comparacion_totales['diferencia'].ne(0).sum())
print(f"Total según cabeceras: {pedidos['total'].sum():,.2f} €")
print(f"Total según líneas: {ventas['importe_venta'].sum():,.2f} €")
display(comparacion_totales.head(10))

Pedidos con diferencia: 100
Total según cabeceras: 12,425.00 €
Total según líneas: 29,650.00 €


,pedido_id,total,total_lineas,diferencia
0,1,50.00,90,-40.00
1,2,51.50,192,-140.50
2,3,53.00,306,-253.00
3,4,54.50,432,-377.50
4,5,56.00,570,-514.00
5,6,57.50,120,-62.50
6,7,59.00,252,-193.00
7,8,60.50,396,-335.50
8,9,62.00,552,-490.00
9,10,63.50,720,-656.50


## 2. Ventas por categoría y mes
La palabra «ventas» puede referirse a unidades o a ingresos. Mostramos las dos medidas.

- `groupby(['categoria', 'mes'])` reúne las líneas de la misma categoría y mes.
- `agg` calcula varias medidas para cada grupo.
- `as_index=False` conserva las claves de agrupación como columnas.

La solución original que sumaba `cantidad` es correcta para **unidades vendidas**. Para calcular **ingresos**, sumamos `importe_venta`.

In [ ]:
ventas_por_categoria_mes = (
    ventas.groupby(['categoria', 'mes'], as_index=False)
    .agg(
        unidades_vendidas=('cantidad', 'sum'),
        ingresos=('importe_venta', 'sum')
    )
    .sort_values(['mes', 'categoria'])
    .reset_index(drop=True)
)
display(ventas_por_categoria_mes)

,categoria,mes,unidades_vendidas,ingresos
0,Electrónica,2024-01,91,3520
1,Hogar,2024-01,91,3920
2,Ropa,2024-01,91,3540
3,Electrónica,2024-02,81,3174
4,Hogar,2024-02,79,3434
5,Ropa,2024-02,77,2982
6,Electrónica,2024-03,61,2350
7,Hogar,2024-03,60,2580
8,Ropa,2024-03,61,2370
9,Electrónica,2024-04,19,574


### Vista alternativa: categorías en filas y meses en columnas
`pivot` cambia la presentación sin modificar los importes. Esta disposición facilita comparar la evolución mensual.

In [ ]:
tabla_ingresos = ventas_por_categoria_mes.pivot(
    index='categoria', columns='mes', values='ingresos'
).fillna(0)
display(tabla_ingresos)
print('Abril solo contiene datos hasta el 9 de abril; no es un mes completo.')

Abril solo contiene datos hasta el 9 de abril; no es un mes completo.


mes,2024-01,2024-02,2024-03,2024-04
categoria,,,,
Electrónica,3520,3174,2350,574
Hogar,3920,3434,2580,552
Ropa,3540,2982,2370,654


## 3. Rentabilidad por producto: ingresos menos costes
Calculamos el **beneficio bruto**, porque solo restamos el coste de los productos:

$$\text{Ingresos de una línea} = \text{cantidad} \times \text{precio unitario}$$
$$\text{Coste de una línea} = \text{cantidad} \times \text{coste unitario}$$
$$\text{Beneficio bruto} = \text{ingresos} - \text{costes}$$
$$\text{Margen bruto (\%)} = 100 \times \frac{\text{beneficio bruto}}{\text{ingresos}}$$

Un beneficio de 40 € sobre ingresos de 100 € supone un margen del 40 %. No es beneficio neto: faltarían otros gastos del negocio. Calculamos el margen sobre los totales de cada producto, sin promediar porcentajes de líneas.

In [ ]:
rentabilidad_producto = (
    ventas.groupby(['producto_id', 'nombre_producto'], as_index=False)
    .agg(
        unidades_vendidas=('cantidad', 'sum'),
        ingresos=('importe_venta', 'sum'),
        costes=('importe_coste', 'sum'),
        beneficio_bruto=('beneficio_bruto', 'sum')
    )
)
# Si los ingresos fueran cero, el porcentaje quedaría sin definir.
denominador = rentabilidad_producto['ingresos'].where(
    rentabilidad_producto['ingresos'].ne(0)
)
rentabilidad_producto['margen_bruto_pct'] = (
    100 * rentabilidad_producto['beneficio_bruto'] / denominador
).round(2)
rentabilidad_producto = rentabilidad_producto.sort_values(
    ['beneficio_bruto', 'producto_id'], ascending=[False, True]
).reset_index(drop=True)
display(rentabilidad_producto)

,producto_id,nombre_producto,unidades_vendidas,ingresos,costes,beneficio_bruto,margen_bruto_pct
0,30,Producto_30,40,2720,1360,1360,50.00
1,25,Producto_25,40,2320,1160,1160,50.00
2,29,Producto_29,32,2112,1056,1056,50.00
3,20,Producto_20,40,1920,960,960,50.00
4,24,Producto_24,32,1792,896,896,50.00
5,28,Producto_28,24,1536,768,768,50.00
6,15,Producto_15,40,1520,760,760,50.00
7,19,Producto_19,32,1472,736,736,50.00
8,23,Producto_23,24,1296,648,648,50.00
9,10,Producto_10,45,1260,630,630,50.00


## 4. Los cinco clientes con mayor gasto total
### A. Solución directa usando `pedidos.total`
Esta es la solución literal si consideramos el campo `total` como el importe oficial del pedido. Agrupamos los pedidos por cliente, sumamos sus totales y añadimos sus atributos descriptivos.

En estos datos todos los pedidos tienen un identificador único, así que `nunique` cuenta los pedidos distintos. Para resolver posibles empates de gasto, ordenamos después por identificador de cliente.

In [ ]:
gasto_clientes_cabecera = (
    pedidos.groupby('cliente_id', as_index=False)
    .agg(
        numero_pedidos=('pedido_id', 'nunique'),
        gasto_total=('total', 'sum')
    )
    .merge(clientes, on='cliente_id', validate='one_to_one')
)
top5_cabecera = (
    gasto_clientes_cabecera
    .sort_values(['gasto_total', 'cliente_id'], ascending=[False, True])
    .head(5)
    .reset_index(drop=True)
)
display(top5_cabecera[
    ['cliente_id', 'nombre', 'ciudad', 'numero_pedidos', 'gasto_total']
])

,cliente_id,nombre,ciudad,numero_pedidos,gasto_total
0,20,Cliente_20,Córdoba,5,692.50
1,19,Cliente_19,Granada,5,685.00
2,18,Cliente_18,Sevilla,5,677.50
3,17,Cliente_17,Málaga,5,670.00
4,16,Cliente_16,Córdoba,5,662.50


### B. Solución coherente con los ingresos de las líneas
Si queremos que el gasto total coincida con los ingresos calculados en los ejercicios anteriores, sumamos `importe_venta` por cliente.

**Usamos `nunique`, no el número de filas:** un pedido con tres líneas debe contar como un pedido, no como tres.

Las dos soluciones responden a fuentes distintas del importe. En un sistema real investigaríamos y resolveríamos la diferencia antes de utilizar los resultados para tomar decisiones.

In [ ]:
gasto_clientes_lineas = (
    ventas.groupby('cliente_id', as_index=False)
    .agg(
        numero_pedidos=('pedido_id', 'nunique'),
        gasto_total=('importe_venta', 'sum')
    )
    .merge(clientes, on='cliente_id', validate='one_to_one')
)
top5_lineas = (
    gasto_clientes_lineas
    .sort_values(['gasto_total', 'cliente_id'], ascending=[False, True])
    .head(5)
    .reset_index(drop=True)
)
display(top5_lineas[
    ['cliente_id', 'nombre', 'ciudad', 'numero_pedidos', 'gasto_total']
])

,cliente_id,nombre,ciudad,numero_pedidos,gasto_total
0,10,Cliente_10,Sevilla,5,3020
1,20,Cliente_20,Córdoba,5,2880
2,5,Cliente_5,Málaga,5,2370
3,9,Cliente_9,Málaga,5,2312
4,15,Cliente_15,Granada,5,2280


## Comprobaciones finales
Agrupar los mismos ingresos por categoría, producto o cliente debe conservar su suma. Estas comprobaciones ayudan a detectar pérdidas de registros o duplicaciones durante las uniones.

In [ ]:
from math import isclose

ingresos_totales = ventas['importe_venta'].sum()
assert isclose(ventas_por_categoria_mes['ingresos'].sum(), ingresos_totales)
assert isclose(rentabilidad_producto['ingresos'].sum(), ingresos_totales)
assert isclose(gasto_clientes_lineas['gasto_total'].sum(), ingresos_totales)
assert isclose(gasto_clientes_cabecera['gasto_total'].sum(), pedidos['total'].sum())
assert isclose(
    rentabilidad_producto['beneficio_bruto'].sum(),
    ingresos_totales - ventas['importe_coste'].sum()
)
print('Comprobaciones superadas: las agrupaciones conservan los totales.')
print(f'Ingresos por líneas: {ingresos_totales:,.2f} €')
print(f"Costes de producto: {ventas['importe_coste'].sum():,.2f} €")
print(f"Beneficio bruto: {ventas['beneficio_bruto'].sum():,.2f} €")

Comprobaciones superadas: las agrupaciones conservan los totales.
Ingresos por líneas: 29,650.00 €
Costes de producto: 14,825.00 €
Beneficio bruto: 14,825.00 €


## Ideas clave para recordar

| Concepto | Aplicación en el ejercicio |
|---|---|
| Granularidad | Una fila de `ventas` representa una línea de pedido |
| Hechos | Cantidades, importes y costes de las ventas |
| Dimensiones | Producto y cliente; el tiempo se obtiene de la fecha |
| `merge` | Añade contexto mediante claves comunes |
| `groupby` + `agg` | Resume las medidas por categoría, mes, producto o cliente |
| `nunique` | Cuenta pedidos distintos aunque tengan varias líneas |
| Calidad de datos | Comprueba que cabeceras y líneas tengan importes coherentes |

**Para practicar:** calcula los ingresos por ciudad y mes, identifica la categoría con mayor beneficio bruto y calcula el ticket medio por cliente usando los totales reconstruidos desde las líneas.